# Lecture 12 — OCR / Document AI (Practice)

Этот ноутбук — **практика без оценивания**.  
Цель: собрать OCR-пайплайн «как в жизни»: **preprocess → OCR → постобработка → извлечение полей → метрики**.

Что внутри:
1) Генерация синтетического документа (без скачиваний)  
2) OCR через Tesseract: текст + координаты слов  
3) Препроцессинг: grayscale, threshold, denoise, deskew (упрощённо)  
4) Метрики: CER/WER  
5) Извлечение полей: invoice_no, date, total  
6) Типовые провалы: качество, наклон, контраст, “0/O”, “1/I”, разделители

Если что-то “не взлетело” — отлично. Это и есть жизнь OCR.


In [ ]:
# Install (Colab)
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract opencv-python-headless pillow matplotlib


In [ ]:
import re, math, random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import cv2
import pytesseract
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 140

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(42)

def to_pil(img):
    if isinstance(img, Image.Image):
        return img
    return Image.fromarray(img)

def show(img, title=None):
    plt.figure(figsize=(8,4))
    plt.imshow(np.array(img), cmap="gray" if img.mode == "L" else None)
    if title: plt.title(title)
    plt.axis("off")
    plt.show()


## 1) Генерируем синтетический документ

Смысл: мы не тянем внешние файлы, но получаем документ, похожий на чек/инвойс.


In [ ]:
def make_invoice(seed=1, angle_deg=0.0, noise=0.0, low_contrast=False) -> Tuple[Image.Image, Dict[str,str]]:
    rng = np.random.RandomState(seed)
    W, H = 900, 520
    bg = 245 if not low_contrast else 220
    fg = 10  if not low_contrast else 60

    img = Image.new("L", (W, H), color=bg)
    draw = ImageDraw.Draw(img)

    # Use default font (available everywhere)
    font = ImageFont.load_default()

    invoice_no = f"INV-{rng.randint(10000, 99999)}"
    date = f"2026-0{rng.randint(2, 9)}-{rng.randint(10, 28)}"
    total = f"{rng.randint(12, 199)}.{rng.randint(0, 99):02d}"

    lines = [
        "INVOICE",
        f"Invoice No: {invoice_no}",
        f"Date: {date}",
        "",
        "Items:",
        "  1) Service A    49.90",
        "  2) Service B    19.50",
        "  3) Product C    12.00",
        "",
        f"TOTAL: {total}",
        "Thank you!",
    ]

    y = 30
    for line in lines:
        draw.text((40, y), line, fill=fg, font=font)
        y += 24

    # Draw a simple box around TOTAL to mimic forms
    draw.rectangle([35, y-48, 420, y-20], outline=fg, width=1)

    # Rotate (simulate camera angle)
    if abs(angle_deg) > 0.001:
        img = img.rotate(angle_deg, expand=True, fillcolor=bg)

    # Add noise
    if noise > 0:
        arr = np.array(img).astype(np.float32)
        arr += rng.normal(0, noise, size=arr.shape)
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        img = Image.fromarray(arr, mode="L")

    gt = {"invoice_no": invoice_no, "date": date, "total": total}
    return img, gt

img_clean, gt = make_invoice(seed=2, angle_deg=0, noise=0, low_contrast=False)
show(img_clean, "Synthetic invoice (clean)")
print("GT:", gt)


## 2) OCR как есть (без препроцессинга)

- `image_to_string` — распознанный текст  
- `image_to_data` — слова + координаты (детекция слов)


In [ ]:
def ocr_text(img: Image.Image, lang="eng") -> str:
    config = "--oem 3 --psm 6"
    return pytesseract.image_to_string(img, lang=lang, config=config)

txt_raw = ocr_text(img_clean)
print(txt_raw)


In [ ]:
def ocr_words_with_boxes(img: Image.Image, lang="eng"):
    config = "--oem 3 --psm 6"
    data = pytesseract.image_to_data(img, lang=lang, config=config, output_type=pytesseract.Output.DICT)
    words = []
    n = len(data["text"])
    for i in range(n):
        w = data["text"][i].strip()
        if not w:
            continue
        x, y, wdt, hgt = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
        conf = float(data["conf"][i]) if str(data["conf"][i]).strip() != "-1" else -1.0
        words.append({"text": w, "bbox": (x, y, x+wdt, y+hgt), "conf": conf})
    return words

words = ocr_words_with_boxes(img_clean)
print("words:", len(words), "sample:", words[:5])


In [ ]:
def draw_word_boxes(img: Image.Image, words, min_conf=0):
    rgb = img.convert("RGB")
    draw = ImageDraw.Draw(rgb)
    for w in words:
        if w["conf"] < min_conf:
            continue
        x1,y1,x2,y2 = w["bbox"]
        draw.rectangle([x1,y1,x2,y2], outline=(255,0,0), width=1)
    return rgb

boxed = draw_word_boxes(img_clean, words, min_conf=30)
show(boxed, "Word boxes (conf>=30)")


## 3) Препроцессинг для OCR

Здесь нет “идеального рецепта”. Но есть рабочие приёмы:
- grayscale (если нужно)
- adaptive threshold / Otsu threshold
- denoise (median / bilateral)
- deskew (упрощённо: по линии текста)

Важно: препроцессинг должен помогать именно вашему источнику.


In [ ]:
def preprocess_for_ocr(img: Image.Image):
    arr = np.array(img)
    if arr.ndim == 3:
        arr = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)

    # Denoise a bit
    arr = cv2.medianBlur(arr, 3)

    # Binarize (Otsu)
    _, th = cv2.threshold(arr, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return Image.fromarray(th)

img_pre = preprocess_for_ocr(img_clean)
show(img_pre, "Preprocessed (Otsu + median)")
print(ocr_text(img_pre))


## 4) CER и WER

- **CER (Character Error Rate)**: (edit_distance chars) / (len(reference))  
- **WER (Word Error Rate)**: (edit_distance words) / (len(reference words))

Это базовые метрики качества OCR, особенно когда важен “точный текст”.


In [ ]:
def edit_distance(a, b):
    # Levenshtein (DP)
    n, m = len(a), len(b)
    dp = np.zeros((n+1, m+1), dtype=np.int32)
    dp[:,0] = np.arange(n+1)
    dp[0,:] = np.arange(m+1)
    for i in range(1, n+1):
        for j in range(1, m+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i,j] = min(dp[i-1,j] + 1, dp[i,j-1] + 1, dp[i-1,j-1] + cost)
    return int(dp[n,m])

def cer(ref: str, hyp: str) -> float:
    ref = ref or ""
    hyp = hyp or ""
    return 0.0 if len(ref) == 0 else edit_distance(ref, hyp) / len(ref)

def wer(ref: str, hyp: str) -> float:
    r = ref.split()
    h = hyp.split()
    return 0.0 if len(r) == 0 else edit_distance(r, h) / len(r)

ref_text = "Invoice No: " + gt["invoice_no"]
hyp_text = ocr_text(img_pre).replace("\n", " ")
print("CER:", round(cer(ref_text, hyp_text), 4))
print("WER:", round(wer(ref_text, hyp_text), 4))


## 5) Извлечение полей (Document AI на минималках)

Document AI в реальности — это комбинация:
- OCR (текст + координаты)
- правила/регексы для полей
- нормализация (дата, суммы)
- иногда ML-модель для layout и классификации строк

Здесь сделаем rules-first: regex + нормализация.


In [ ]:
def normalize_date(s: str) -> str:
    # Expect YYYY-MM-DD, but allow common OCR mess like YYYY-0X-1O
    s = s.strip()
    s = s.replace("O", "0").replace("I", "1").replace("l", "1")
    m = re.search(r"(20\d{2})[-/.](\d{1,2})[-/.](\d{1,2})", s)
    if not m:
        return ""
    y, mo, d = m.group(1), int(m.group(2)), int(m.group(3))
    return f"{y}-{mo:02d}-{d:02d}"

def normalize_money(s: str) -> str:
    s = s.strip()
    s = s.replace("O", "0").replace(",", ".")
    m = re.search(r"(\d+\.?\d*)", s)
    if not m:
        return ""
    # keep 2 decimals if possible
    val = float(m.group(1))
    return f"{val:.2f}"

def extract_fields(text: str) -> Dict[str,str]:
    # invoice number
    inv = ""
    m = re.search(r"Invoice\s*No\s*[:\-]\s*([A-Z]{2,5}[- ]?\d{4,6})", text, re.IGNORECASE)
    if m:
        inv = m.group(1).replace(" ", "")
    # date
    date = ""
    m = re.search(r"Date\s*[:\-]\s*([^\n]+)", text, re.IGNORECASE)
    if m:
        date = normalize_date(m.group(1))
    # total
    total = ""
    m = re.search(r"TOTAL\s*[:\-]\s*([^\n]+)", text, re.IGNORECASE)
    if m:
        total = normalize_money(m.group(1))
    return {"invoice_no": inv, "date": date, "total": total}

ocr_out = ocr_text(img_pre)
fields = extract_fields(ocr_out)
print("Extracted:", fields)
print("GT:", gt)


## 6) Типовые проблемы OCR (мини-демо)

Сделаем 3 плохих условия:
- низкий контраст
- наклон
- шум

И посмотрим, как растёт CER/WER и как ломаются поля.


In [ ]:
cases = [
    ("low_contrast", make_invoice(seed=3, low_contrast=True)[0]),
    ("rotated_8deg", make_invoice(seed=4, angle_deg=8)[0]),
    ("noise_sigma15", make_invoice(seed=5, noise=15)[0]),
]

for name, img in cases:
    img_pre = preprocess_for_ocr(img)
    txt = ocr_text(img_pre)
    f = extract_fields(txt)
    print("\nCASE:", name)
    print("fields:", f)
